In [1]:
# # STEP 03.1 — CROSS-MODAL DATASET PREPARATION

# ## Objective

# Prepare the validated cross-modal pair dataset for model training.

# This step verifies:

# - Pair dataset integrity
# - Source-target dimensional compatibility
# - Positive-negative balance
# - Metadata consistency
# - Train-validation separation
# - Reproducible splitting
# - PyTorch dataset compatibility

# ### Methodological Principle

# The dataset represents knowledge-level cross-modal relationships.

# Each row represents:

# Source Representation ↔ Target Representation

# The pairs do NOT represent measurements from the same patient.

# No patient-level correspondence is assumed.

In [2]:
# ============================================================
# CELL 2 — IMPORTS AND PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

print("=" * 100)
print("STEP 03.1 — CROSS-MODAL DATASET PREPARATION")
print("=" * 100)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

# ------------------------------------------------------------
# PHASE 02 FINAL DATASET
# ------------------------------------------------------------

PAIR_DATASET_DIR = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
    / "pairs"
)

# ------------------------------------------------------------
# PHASE 03 OUTPUT DIRECTORIES
# ------------------------------------------------------------

PHASE03_DIR = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "03_cross_modal_training"
)

REPORT_DIR = PHASE03_DIR / "reports"
DATASET_OUTPUT_DIR = PHASE03_DIR / "prepared_dataset"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nProject Root:")
print(PROJECT_ROOT)

print("\nPhase 02 Pair Dataset Directory:")
print(PAIR_DATASET_DIR)

print("\nPhase 03 Output Directory:")
print(PHASE03_DIR)

print("\n✓ Initialization completed")

STEP 03.1 — CROSS-MODAL DATASET PREPARATION

Project Root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 02 Pair Dataset Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs

Phase 03 Output Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_training

✓ Initialization completed


In [3]:
# ============================================================
# CELL 3 — INSPECT FINAL PHASE 02 PAIR DATASET
# ============================================================

print("=" * 100)
print("PHASE 02 FINAL PAIR DATASET INVENTORY")
print("=" * 100)

if not PAIR_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Pair dataset directory not found:\n{PAIR_DATASET_DIR}"
    )

pair_files = sorted(PAIR_DATASET_DIR.iterdir())

print("\nAvailable files:\n")

for file in pair_files:
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)

        print(
            f"{file.name:<65} "
            f"{size_mb:.4f} MB"
        )

print("\nTotal files found:", len(pair_files))

print("\nExpected Phase 03 inputs:")
print("- Final pair dataset (.npz)")
print("- Pair metadata (.csv)")
print("- Dataset split information if available")

print("\n✓ Dataset inventory completed")

PHASE 02 FINAL PAIR DATASET INVENTORY

Available files:

02_04_positive_cross_modal_embedding_pairs.npz                    0.0362 MB
02_04_positive_cross_modal_pair_metadata.csv                      0.0126 MB
02_05_negative_cross_modal_embedding_pairs.npz                    0.0360 MB
02_05_negative_cross_modal_pair_metadata.csv                      0.0110 MB

Total files found: 6

Expected Phase 03 inputs:
- Final pair dataset (.npz)
- Pair metadata (.csv)
- Dataset split information if available

✓ Dataset inventory completed


In [4]:
# ============================================================
# CELL 4 — LOAD FINAL CROSS-MODAL PAIR DATASET
# ============================================================

print("=" * 100)
print("LOADING FINAL CROSS-MODAL PAIR DATASET")
print("=" * 100)

# ------------------------------------------------------------
# FIND NPZ DATASET FILE
# ------------------------------------------------------------

npz_files = list(PAIR_DATASET_DIR.glob("*.npz"))

if len(npz_files) == 0:
    raise FileNotFoundError(
        "No .npz pair dataset found in the Phase 02 pairs directory."
    )

print("\nAvailable NPZ files:")

for file in npz_files:
    print("-", file.name)

# Select the most recent / final dataset
PAIR_DATASET_PATH = npz_files[-1]

print("\nSelected dataset:")
print(PAIR_DATASET_PATH)

# ------------------------------------------------------------
# LOAD NPZ
# ------------------------------------------------------------

pair_data = np.load(
    PAIR_DATASET_PATH,
    allow_pickle=True
)

print("\nAvailable arrays inside dataset:")

for key in pair_data.files:
    print("-", key)

print("\n✓ Pair dataset loaded successfully")

LOADING FINAL CROSS-MODAL PAIR DATASET

Available NPZ files:
- 02_04_positive_cross_modal_embedding_pairs.npz
- 02_05_negative_cross_modal_embedding_pairs.npz

Selected dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\02_05_negative_cross_modal_embedding_pairs.npz

Available arrays inside dataset:
- source_embeddings
- target_embeddings

✓ Pair dataset loaded successfully


In [6]:
# ============================================================
# CELL 5 — EXTRACT AND VERIFY DATASET STRUCTURE
# ============================================================

print("=" * 100)
print("CROSS-MODAL PAIR DATASET STRUCTURAL VERIFICATION")
print("=" * 100)

# ------------------------------------------------------------
# EXTRACT ARRAYS
# ------------------------------------------------------------

source_embeddings = pair_data["source_embeddings"]
target_embeddings = pair_data["target_embeddings"]
pair_labels = pair_data["pair_labels"]

# ------------------------------------------------------------
# DISPLAY SHAPES
# ------------------------------------------------------------

print("\nSOURCE EMBEDDINGS")
print("Shape :", source_embeddings.shape)
print("Dtype :", source_embeddings.dtype)

print("\nTARGET EMBEDDINGS")
print("Shape :", target_embeddings.shape)
print("Dtype :", target_embeddings.dtype)

print("\nPAIR LABELS")
print("Shape :", pair_labels.shape)
print("Dtype :", pair_labels.dtype)

# ------------------------------------------------------------
# STRUCTURAL CHECKS
# ------------------------------------------------------------

n_source = len(source_embeddings)
n_target = len(target_embeddings)
n_labels = len(pair_labels)

assert n_source == n_target, (
    "Source and target pair counts do not match."
)

assert n_source == n_labels, (
    "Embedding pairs and labels do not match."
)

assert source_embeddings.shape[1] == 32, (
    "Source embeddings are not 32-dimensional."
)

assert target_embeddings.shape[1] == 32, (
    "Target embeddings are not 32-dimensional."
)

# ------------------------------------------------------------
# NUMERICAL CHECKS
# ------------------------------------------------------------

assert not np.isnan(source_embeddings).any()
assert not np.isnan(target_embeddings).any()

assert not np.isinf(source_embeddings).any()
assert not np.isinf(target_embeddings).any()

print("\n" + "=" * 100)
print("STRUCTURAL VERIFICATION SUMMARY")
print("=" * 100)

print(f"Total cross-modal pairs        : {n_source}")
print(f"Source embedding dimension     : {source_embeddings.shape[1]}")
print(f"Target embedding dimension     : {target_embeddings.shape[1]}")
print(f"Pair-label correspondence      : VERIFIED")
print(f"Numerical integrity            : VERIFIED")

print("\n✓ DATASET STRUCTURE VALID")

INSPECTING FINAL PAIR DATASET

Available keys inside pair dataset:

Key   : source_embeddings
Shape : (174, 32)
Dtype : float32
------------------------------------------------------------
Key   : target_embeddings
Shape : (174, 32)
Dtype : float32
------------------------------------------------------------


In [12]:
# ============================================================
# CELL 5A — LOAD EXISTING TRAIN AND VALIDATION PAIR DATASETS
# ============================================================

print("=" * 100)
print("LOADING EXISTING TRAIN AND VALIDATION PAIR DATASETS")
print("=" * 100)

# ------------------------------------------------------------
# Dataset Directory
# ------------------------------------------------------------

FINAL_DATASET_DIR = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
    / "pairs"
    / "final_dataset"
)

# ------------------------------------------------------------
# Existing Phase 02.7 Split Files
# ------------------------------------------------------------

TRAIN_PAIR_PATH = (
    FINAL_DATASET_DIR
    / "02_07_train_cross_modal_pairs.npz"
)

VALIDATION_PAIR_PATH = (
    FINAL_DATASET_DIR
    / "02_07_validation_cross_modal_pairs.npz"
)

# ------------------------------------------------------------
# Verify Files Exist
# ------------------------------------------------------------

print("\nTrain dataset:")
print(TRAIN_PAIR_PATH)
print("Exists:", TRAIN_PAIR_PATH.exists())

print("\nValidation dataset:")
print(VALIDATION_PAIR_PATH)
print("Exists:", VALIDATION_PAIR_PATH.exists())

# ------------------------------------------------------------
# Load Datasets
# ------------------------------------------------------------

train_data = np.load(TRAIN_PAIR_PATH)
validation_data = np.load(VALIDATION_PAIR_PATH)

# ------------------------------------------------------------
# Inspect Available Keys
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TRAIN DATASET KEYS")
print("=" * 100)

for key in train_data.files:
    print(f"{key} : {train_data[key].shape}")

print("\n" + "=" * 100)
print("VALIDATION DATASET KEYS")
print("=" * 100)

for key in validation_data.files:
    print(f"{key} : {validation_data[key].shape}")

print("\n" + "=" * 100)
print("✓ EXISTING TRAIN / VALIDATION DATASETS LOADED")
print("✓ NO NEW DATA SPLIT CREATED")
print("✓ PHASE 02.7 SPLIT PRESERVED")
print("=" * 100)

LOADING EXISTING TRAIN AND VALIDATION PAIR DATASETS

Train dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset\02_07_train_cross_modal_pairs.npz
Exists: True

Validation dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset\02_07_validation_cross_modal_pairs.npz
Exists: True

TRAIN DATASET KEYS
source_embeddings : (278, 32)
target_embeddings : (278, 32)
pair_labels : (278,)

VALIDATION DATASET KEYS
source_embeddings : (70, 32)
target_embeddings : (70, 32)
pair_labels : (70,)

✓ EXISTING TRAIN / VALIDATION DATASETS LOADED
✓ NO NEW DATA SPLIT CREATED
✓ PHASE 02.7 SPLIT PRESERVED


In [11]:
# ============================================================
# CELL 6 — EXTRACT TRAINING AND VALIDATION ARRAYS
# ============================================================

print("=" * 100)
print("EXTRACTING TRAINING AND VALIDATION ARRAYS")
print("=" * 100)

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_source_train = train_data["source_embeddings"]
X_target_train = train_data["target_embeddings"]
y_train = train_data["pair_labels"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_source_val = validation_data["source_embeddings"]
X_target_val = validation_data["target_embeddings"]
y_val = validation_data["pair_labels"]

# ------------------------------------------------------------
# Display Shapes
# ------------------------------------------------------------

print("\nTRAINING DATA")
print("-" * 60)
print(f"Source embeddings : {X_source_train.shape}")
print(f"Target embeddings : {X_target_train.shape}")
print(f"Pair labels       : {y_train.shape}")

print("\nVALIDATION DATA")
print("-" * 60)
print(f"Source embeddings : {X_source_val.shape}")
print(f"Target embeddings : {X_target_val.shape}")
print(f"Pair labels       : {y_val.shape}")

# ------------------------------------------------------------
# Structural Verification
# ------------------------------------------------------------

assert len(X_source_train) == len(X_target_train) == len(y_train)
assert len(X_source_val) == len(X_target_val) == len(y_val)

assert X_source_train.shape[1] == 32
assert X_target_train.shape[1] == 32

assert X_source_val.shape[1] == 32
assert X_target_val.shape[1] == 32

print("\n" + "=" * 100)
print("✓ TRAINING STRUCTURE VERIFIED")
print("✓ VALIDATION STRUCTURE VERIFIED")
print("✓ ALL EMBEDDINGS CONFIRMED AS 32-D")
print("✓ SOURCE-TARGET-LABEL CORRESPONDENCE PRESERVED")
print("=" * 100)

EXTRACTING TRAINING AND VALIDATION ARRAYS

TRAINING DATA
------------------------------------------------------------
Source embeddings : (278, 32)
Target embeddings : (278, 32)
Pair labels       : (278,)

VALIDATION DATA
------------------------------------------------------------
Source embeddings : (70, 32)
Target embeddings : (70, 32)
Pair labels       : (70,)

✓ TRAINING STRUCTURE VERIFIED
✓ VALIDATION STRUCTURE VERIFIED
✓ ALL EMBEDDINGS CONFIRMED AS 32-D
✓ SOURCE-TARGET-LABEL CORRESPONDENCE PRESERVED


In [13]:
# ============================================================
# CELL 7 — VALIDATE TRAIN / VALIDATION PAIR LABEL DISTRIBUTION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("VALIDATING TRAIN / VALIDATION PAIR LABEL DISTRIBUTION")
print("=" * 100)

# ------------------------------------------------------------
# Function to calculate pair label statistics
# ------------------------------------------------------------

def get_label_distribution(labels, dataset_name):

    unique, counts = np.unique(labels, return_counts=True)

    distribution = {
        int(label): int(count)
        for label, count in zip(unique, counts)
    }

    total = len(labels)

    positive_count = distribution.get(1, 0)
    negative_count = distribution.get(0, 0)

    positive_percentage = (
        positive_count / total * 100
        if total > 0 else 0
    )

    negative_percentage = (
        negative_count / total * 100
        if total > 0 else 0
    )

    return {
        "Dataset": dataset_name,
        "Total_Pairs": total,
        "Positive_Pairs": positive_count,
        "Negative_Pairs": negative_count,
        "Positive_Percentage": positive_percentage,
        "Negative_Percentage": negative_percentage
    }


# ------------------------------------------------------------
# Calculate distributions
# ------------------------------------------------------------

train_distribution = get_label_distribution(
    y_train,
    "Training"
)

validation_distribution = get_label_distribution(
    y_val,
    "Validation"
)

# ------------------------------------------------------------
# Create Summary Table
# ------------------------------------------------------------

distribution_summary = pd.DataFrame([
    train_distribution,
    validation_distribution
])

print("\nPAIR LABEL DISTRIBUTION")
print("-" * 100)

display(distribution_summary)

# ------------------------------------------------------------
# Detailed Output
# ------------------------------------------------------------

for _, row in distribution_summary.iterrows():

    print(f"\n{row['Dataset'].upper()}")
    print("-" * 50)

    print(f"Total pairs       : {row['Total_Pairs']}")
    print(f"Positive pairs    : {row['Positive_Pairs']} "
          f"({row['Positive_Percentage']:.2f}%)")
    print(f"Negative pairs    : {row['Negative_Pairs']} "
          f"({row['Negative_Percentage']:.2f}%)")


# ------------------------------------------------------------
# Basic Validation
# ------------------------------------------------------------

assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})

print("\n" + "=" * 100)
print("✓ TRAIN LABEL DISTRIBUTION VALIDATED")
print("✓ VALIDATION LABEL DISTRIBUTION VALIDATED")
print("✓ POSITIVE / NEGATIVE RELATIONSHIP LABELS VERIFIED")
print("=" * 100)

VALIDATING TRAIN / VALIDATION PAIR LABEL DISTRIBUTION

PAIR LABEL DISTRIBUTION
----------------------------------------------------------------------------------------------------


,Dataset,Total_Pairs,Positive_Pairs,Negative_Pairs,Positive_Percentage,Negative_Percentage
0,Training,278,139,139,50.0,50.0
1,Validation,70,35,35,50.0,50.0



TRAINING
--------------------------------------------------
Total pairs       : 278
Positive pairs    : 139 (50.00%)
Negative pairs    : 139 (50.00%)

VALIDATION
--------------------------------------------------
Total pairs       : 70
Positive pairs    : 35 (50.00%)
Negative pairs    : 35 (50.00%)

✓ TRAIN LABEL DISTRIBUTION VALIDATED
✓ VALIDATION LABEL DISTRIBUTION VALIDATED
✓ POSITIVE / NEGATIVE RELATIONSHIP LABELS VERIFIED


In [14]:
# ============================================================
# CELL 9 — EXACT TRAIN / VALIDATION PAIR OVERLAP CHECK
# ============================================================

print("=" * 100)
print("TRAIN / VALIDATION EXACT PAIR OVERLAP CHECK")
print("=" * 100)

# ------------------------------------------------------------
# Create exact pair signatures
# ------------------------------------------------------------

def create_pair_signatures(source, target, labels):
    """
    Creates a numerical signature for each complete pair.

    A signature includes:
    source embedding +
    target embedding +
    relationship label

    This allows us to detect exact duplicate relationships.
    """

    signatures = set()

    for s, t, label in zip(source, target, labels):

        signature = (
            tuple(np.round(s, 6)),
            tuple(np.round(t, 6)),
            int(label)
        )

        signatures.add(signature)

    return signatures


# ------------------------------------------------------------
# Generate signatures
# ------------------------------------------------------------

train_signatures = create_pair_signatures(
    X_source_train,
    X_target_train,
    y_train
)

validation_signatures = create_pair_signatures(
    X_source_val,
    X_target_val,
    y_val
)

# ------------------------------------------------------------
# Find exact overlap
# ------------------------------------------------------------

overlap = train_signatures.intersection(validation_signatures)

print(f"\nUnique train relationships      : {len(train_signatures)}")
print(f"Unique validation relationships : {len(validation_signatures)}")
print(f"Exact overlapping relationships : {len(overlap)}")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if len(overlap) == 0:

    print("\n✓ NO EXACT PAIR DUPLICATION DETECTED")
    print("✓ TRAIN AND VALIDATION RELATIONSHIPS ARE SEPARATED")

else:

    print("\n⚠ EXACT PAIR OVERLAP DETECTED")
    print("⚠ REVIEW PAIR SPLITTING STRATEGY")

print("\n" + "=" * 100)
print("EXACT PAIR OVERLAP CHECK COMPLETED")
print("=" * 100)

TRAIN / VALIDATION EXACT PAIR OVERLAP CHECK

Unique train relationships      : 278
Unique validation relationships : 70
Exact overlapping relationships : 0

✓ NO EXACT PAIR DUPLICATION DETECTED
✓ TRAIN AND VALIDATION RELATIONSHIPS ARE SEPARATED

EXACT PAIR OVERLAP CHECK COMPLETED


In [15]:
# ============================================================
# CELL 10 — REPRESENTATION REUSE ACROSS SPLITS
# ============================================================

print("=" * 100)
print("REPRESENTATION REUSE ACROSS TRAIN / VALIDATION")
print("=" * 100)


def create_embedding_signatures(embeddings):
    """Create hashable signatures for embedding vectors."""
    
    return {
        tuple(np.round(embedding, 6))
        for embedding in embeddings
    }


# ------------------------------------------------------------
# Create embedding sets
# ------------------------------------------------------------

train_source_set = create_embedding_signatures(X_source_train)
train_target_set = create_embedding_signatures(X_target_train)

val_source_set = create_embedding_signatures(X_source_val)
val_target_set = create_embedding_signatures(X_target_val)


# ------------------------------------------------------------
# Calculate overlaps
# ------------------------------------------------------------

source_overlap = (
    train_source_set.intersection(val_source_set)
)

target_overlap = (
    train_target_set.intersection(val_target_set)
)

all_train_embeddings = (
    train_source_set.union(train_target_set)
)

all_val_embeddings = (
    val_source_set.union(val_target_set)
)

overall_embedding_overlap = (
    all_train_embeddings.intersection(all_val_embeddings)
)


# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

print(f"\nTrain unique source embeddings      : {len(train_source_set)}")
print(f"Validation unique source embeddings : {len(val_source_set)}")
print(f"Source representation overlap       : {len(source_overlap)}")

print()

print(f"Train unique target embeddings      : {len(train_target_set)}")
print(f"Validation unique target embeddings : {len(val_target_set)}")
print(f"Target representation overlap       : {len(target_overlap)}")

print()

print(f"Overall unique train representations      : {len(all_train_embeddings)}")
print(f"Overall unique validation representations : {len(all_val_embeddings)}")
print(f"Overall representation overlap            : {len(overall_embedding_overlap)}")


print("\n" + "=" * 100)
print("METHODOLOGICAL INTERPRETATION")
print("=" * 100)

if len(overall_embedding_overlap) == 0:
    print("""
✓ No representation reuse detected across train and validation.

The split is representation-disjoint, which provides stronger
validation independence.
""")
else:
    print("""
⚠ Some learned representations are reused across train and validation.

This does NOT imply patient-level leakage because no patient
correspondence exists across modalities.

However, validation independence should be interpreted as
pair-level rather than fully representation-disjoint.
""")

REPRESENTATION REUSE ACROSS TRAIN / VALIDATION

Train unique source embeddings      : 135
Validation unique source embeddings : 53
Source representation overlap       : 48

Train unique target embeddings      : 275
Validation unique target embeddings : 70
Target representation overlap       : 3

Overall unique train representations      : 407
Overall unique validation representations : 123
Overall representation overlap            : 52

METHODOLOGICAL INTERPRETATION

⚠ Some learned representations are reused across train and validation.

This does NOT imply patient-level leakage because no patient
correspondence exists across modalities.

However, validation independence should be interpreted as
pair-level rather than fully representation-disjoint.



In [16]:
# ============================================================
# CELL 10A — QUANTIFY REPRESENTATION OVERLAP SEVERITY
# ============================================================

print("=" * 100)
print("REPRESENTATION OVERLAP SEVERITY ASSESSMENT")
print("=" * 100)

# ------------------------------------------------------------
# Calculate overlap percentages
# ------------------------------------------------------------

source_overlap_percentage = (
    len(source_overlap) / len(val_source_set) * 100
    if len(val_source_set) > 0 else 0
)

target_overlap_percentage = (
    len(target_overlap) / len(val_target_set) * 100
    if len(val_target_set) > 0 else 0
)

overall_overlap_percentage = (
    len(overall_embedding_overlap) / len(all_val_embeddings) * 100
    if len(all_val_embeddings) > 0 else 0
)

overlap_severity = pd.DataFrame([
    {
        "Representation_Type": "Source",
        "Validation_Unique_Representations": len(val_source_set),
        "Overlapping_With_Train": len(source_overlap),
        "Overlap_Percentage": source_overlap_percentage
    },
    {
        "Representation_Type": "Target",
        "Validation_Unique_Representations": len(val_target_set),
        "Overlapping_With_Train": len(target_overlap),
        "Overlap_Percentage": target_overlap_percentage
    },
    {
        "Representation_Type": "Overall",
        "Validation_Unique_Representations": len(all_val_embeddings),
        "Overlapping_With_Train": len(overall_embedding_overlap),
        "Overlap_Percentage": overall_overlap_percentage
    }
])

print("\nREPRESENTATION OVERLAP SUMMARY")
print("-" * 100)

display(overlap_severity)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("METHODOLOGICAL ASSESSMENT")
print("=" * 100)

for _, row in overlap_severity.iterrows():

    percentage = row["Overlap_Percentage"]

    if percentage == 0:
        severity = "NONE — Fully representation-disjoint"
    elif percentage < 20:
        severity = "LOW"
    elif percentage < 50:
        severity = "MODERATE"
    else:
        severity = "HIGH"

    print(
        f"\n{row['Representation_Type']}: "
        f"{percentage:.2f}% overlap → {severity}"
    )

print("\nImportant:")
print("This measures representation reuse, NOT patient identity leakage.")
print("The current split remains pair-disjoint.")
print("Representation overlap affects the strength of validation independence.")

print("\n" + "=" * 100)
print("✓ REPRESENTATION OVERLAP SEVERITY QUANTIFIED")
print("=" * 100)


REPRESENTATION OVERLAP SEVERITY ASSESSMENT

REPRESENTATION OVERLAP SUMMARY
----------------------------------------------------------------------------------------------------


,Representation_Type,Validation_Unique_Representations,Overlapping_With_Train,Overlap_Percentage
0,Source,53,48,90.566038
1,Target,70,3,4.285714
2,Overall,123,52,42.276423



METHODOLOGICAL ASSESSMENT

Source: 90.57% overlap → HIGH

Target: 4.29% overlap → LOW

Overall: 42.28% overlap → MODERATE

Important:
This measures representation reuse, NOT patient identity leakage.
The current split remains pair-disjoint.
Representation overlap affects the strength of validation independence.

✓ REPRESENTATION OVERLAP SEVERITY QUANTIFIED
